In [1]:
using Ferrite
using LinearAlgebra
using SparseArrays
using Tensors
using ForwardDiff
using PhaseFieldFracture # 自己个人的Module

# ==========================================
# 1. 测试环境初始化
# ==========================================
grid = generate_grid(Quadrilateral, (1, 1)) # 单单元网格

# 独立的位移场 DofHandler
dh_u = DofHandler(grid)
add!(dh_u, :u, Lagrange{RefQuadrilateral, 1}()^2)
close!(dh_u)

# 独立的相场 DofHandler
dh_d = DofHandler(grid)
add!(dh_d, :d, Lagrange{RefQuadrilateral, 1}())
close!(dh_d)

# 积分点与 CellValues 准备
qr = QuadratureRule{RefQuadrilateral}(2)
cv_u = CellValues(qr, Lagrange{RefQuadrilateral, 1}()^2)
cv_d = CellValues(qr, Lagrange{RefQuadrilateral, 1}())

# 实例化材料参数
mat = PhaseFieldMaterial(
    E = 25840, # 杨氏模量，单位 MPa
    ν = 0.18,
    gc = 0.65, # 断裂能 gc 的单位是 N/mm
    l = 10, # 尺度参数 l 建议不小于两倍网格尺寸 h
    k_tol = 1e-8
)

# ==========================================
# 2. 测试 assemble_u! (非线性 AD 校验)
# ==========================================
println("--- 启动 assemble_u! 测试 ---")

# 构造随机但合理的测试位移场和相场
u_test = randn(ndofs(dh_u)) * 1e-3
d_test = rand(ndofs(dh_d)) * 0.4 # 相场值在 0 ~ 0.4 之间

# 定义残差包装函数以供 ForwardDiff 求导
function get_u_residual(u_eval)
    T = eltype(u_eval)
    R_temp = zeros(T, ndofs(dh_u))
    
    # 构造对应类型的稀疏矩阵模版以通过 start_assemble 的类型检查
    K_float = allocate_matrix(dh_u)
    K_dummy = SparseMatrixCSC{T, Int}(
        K_float.m, K_float.n,
        K_float.colptr, K_float.rowval,
        zeros(T, length(K_float.nzval))
    )
    
    # 执行组装
    assemble_u!(K_dummy, R_temp, dh_u, dh_d, u_eval, d_test, mat, cv_u, cv_d)
    return R_temp
end

# 1) 数值微分计算雅可比 (K_numerical)
K_numerical_u = ForwardDiff.jacobian(get_u_residual, u_test)

# 2) 您的解析函数计算切线刚度 (K_analytical)
K_analytical_sparse_u = allocate_matrix(dh_u)
r_analytical_u = zeros(ndofs(dh_u))
assemble_u!(K_analytical_sparse_u, r_analytical_u, dh_u, dh_d, u_test, d_test, mat, cv_u, cv_d)
K_analytical_u = Matrix(K_analytical_sparse_u)

# 3) 对比误差
u_error = maximum(abs.(K_analytical_u - K_numerical_u))
println("位移场刚度矩阵最大绝对误差: ", u_error)
if u_error < 1e-10
    println("✓ assemble_u! 通过测试！解析切线与数值梯度精确吻合。")
else
    @warn "✗ assemble_u! 存在误差，请检查材料应力导数或 Tensors.jl 的求导逻辑！"
end

# ==========================================
# 3. 测试 assemble_d! (线性系统一致性校验)
# ==========================================
println("\n--- 启动 assemble_d! 测试 ---")

# 构造随机的历史场 H (每个积分点对应一个值，共 4 个积分点)
H_test = rand(getncells(grid) * getnquadpoints(cv_d)) * 10.0

# 分配矩阵和向量
K_d = allocate_matrix(dh_d)
F_d = zeros(ndofs(dh_d))

# 运行相场组装
assemble_d!(K_d, F_d, dh_d, H_test, mat, cv_d)
K_d_dense = Matrix(K_d)

# 校验线性系统的物理特征：
# 1) 刚度矩阵 K_d 是否对称？
symmetry_error = maximum(abs.(K_d_dense - K_d_dense'))
# 2) 刚度矩阵 K_d 是否正定？（所有特征值应当大于零）
eigenvals = eigen(K_d_dense).values
is_pos_def = all(eigenvals .> 0)

println("相场刚度矩阵对称性误差: ", symmetry_error)
println("相场刚度矩阵是否正定: ", is_pos_def, " (最小特征值: $(minimum(eigenvals)))")

if symmetry_error < 1e-10 && is_pos_def
    println("✓ assemble_d! 通过测试！组装的线性系统对称且正定，物理特性正确。")
else
    @warn "✗ assemble_d! 组装出的刚度矩阵物理上不合理，请检查弱形式的项和系数！"
end


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


--- 启动 assemble_u! 测试 ---
位移场刚度矩阵最大绝对误差: 3.637978807091713e-12
✓ assemble_u! 通过测试！解析切线与数值梯度精确吻合。

--- 启动 assemble_d! 测试 ---
相场刚度矩阵对称性误差: 2.220446049250313e-16
相场刚度矩阵是否正定: true (最小特征值: 4.497318851928432)
✓ assemble_d! 通过测试！组装的线性系统对称且正定，物理特性正确。


In [1]:
using Ferrite
using LinearAlgebra
using SparseArrays
using Tensors
using ForwardDiff
using PhaseFieldFracture # 引入个人模块
macauley_plus(x::Real) = x > zero(x) ? x : zero(x)
# ==========================================
# 1. 测试环境初始化
# ==========================================
grid = generate_grid(Quadrilateral, (1, 1)) # 单单元网格

# 联立的未知量 DofHandler（位移 :u + 相场 :d）
dh_a = DofHandler(grid)
add!(dh_a, :u, Lagrange{RefQuadrilateral, 1}()^2)
add!(dh_a, :d, Lagrange{RefQuadrilateral, 1}())
close!(dh_a)

# 积分点与 CellValues 准备
qr = QuadratureRule{RefQuadrilateral}(2)
cv_u = CellValues(qr, Lagrange{RefQuadrilateral, 1}()^2)
cv_d = CellValues(qr, Lagrange{RefQuadrilateral, 1}())

# 历史变量 H_old 初始化 (1单元 * 4积分点)
H_old = zeros(getncells(grid) * getnquadpoints(cv_u))

# 实例化材料参数 (使用您模块中的类型)
mat = PhaseFieldMaterial(
    E = 25840, # 杨氏模量，单位 MPa
    ν = 0.18,
    gc = 0.65, # 断裂能 gc 的单位是 N/mm
    l = 10, # 尺度参数 l 建议不小于两倍网格尺寸 h
    k_tol = 1e-8
)

# ==========================================
# 2. 构造符合 Ferrite 真实映射关系的测试状态向量 a_test
# ==========================================
# 初始化全零的联合未知量向量
a_test = zeros(ndofs(dh_a))

# 安全地提取单元局部自由度映射关系
cell = first(CellIterator(dh_a))
c_dofs = celldofs(cell)
global_u_dofs = c_dofs[Ferrite.dof_range(dh_a, :u)]
global_d_dofs = c_dofs[Ferrite.dof_range(dh_a, :d)]

# 赋以随机但合理的初值（激活非线性与退化机制）
a_test[global_u_dofs] .= randn(length(global_u_dofs)) * 1e-3
a_test[global_d_dofs] .= 0.3 .+ 0.1 * rand(length(global_d_dofs))

# ==========================================
# 3. 运行 assemble_monolithic! 非线性 AD 校验
# ==========================================
println("--- 启动 assemble_monolithic! 测试 ---")

# 定义全局残差包装函数以供 ForwardDiff 求导
function get_a_residual(a_eval)
    T = eltype(a_eval)
    R_temp = zeros(T, ndofs(dh_a))
    
    # 传入 nothing 替代 K_dummy，避免装配器转换干扰自动微分
    assemble_monolithic!(nothing, R_temp, dh_a, a_eval, H_old, mat, cv_u, cv_d)
    return R_temp
end

# 1) 自动微分计算精确的雅可比矩阵 (K_numerical)
K_numerical_a = ForwardDiff.jacobian(get_a_residual, a_test)

# 2) 您的解析函数计算单块切线刚度 (K_analytical)
K_analytical_sparse_a = allocate_matrix(dh_a)
r_analytical_a = zeros(ndofs(dh_a))
assemble_monolithic!(K_analytical_sparse_a, r_analytical_a, dh_a, a_test, H_old, mat, cv_u, cv_d)
K_analytical_a = Matrix(K_analytical_sparse_a)

# 3) 对比误差
a_error = maximum(abs.(K_analytical_a - K_numerical_a))
println("\n================ AD 联立测试结果 ================")
println("联立方程组刚度矩阵最大绝对误差: ", a_error)

if a_error < 1e-10
    println("✓ assemble_monolithic! 通过测试！单块切线与数值梯度精确吻合。")
else
    @warn "✗ assemble_monolithic! 仍存在误差，请核对数据！"
    row, col = Tuple(argmax(abs.(K_analytical_a - K_numerical_a)))
    println("最大误差发生在第 $row 行，第 $col 列")
    println("解析值: $(K_analytical_a[row,col])")
    println("AD 数值: $(K_numerical_a[row,col])")
end


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


--- 启动 assemble_monolithic! 测试 ---


ArgumentError: ArgumentError: no valid permutation of dimensions